<img src="../img/viu_logo.png" width="200">

# Aprendizaje por Refuerzo: Proyecto de programación

- **Profesor:** Francisco Muñoz Montoya
- **Autores:** Guillermo Sanchez Garcia, Diego Aguado Sanchez, Alejandro Pequeno Lizcano, Jaume Montanera

### Índice
0. [Introducción](#0-introducción)
1. [Implementación red neuronal](#1-implementación-red-neuronal)
2. [Implementación del agente (DQN)](#2-implementación-del-agente)
3. [Conclusiones](#3-conclusiones)

## 0. Introducción

Consideraciones a tener en cuenta:

- El entorno sobre el que trabajaremos será el indicado en el listado correspondien de cada grupo y el algoritmo que usaremos será _DQN_.

- Para nuestro ejercicio, el requisito mínimo será alcanzado cuando el agente consiga una **media de recompensa por encima de los puntos indicados en el listado por grupos en modo test**. Por ello, esta media de la recompensa se calculará a partir del código de test en la última celda del notebook.

Este proyecto práctico consta de tres partes:

1.   Implementar la red neuronal que se usará en la solución
2.   Implementar las distintas piezas de la solución DQN y probar al menos 3 propuestas diferentes de mejora.
3.   Justificar la respuesta en relación a los resultados obtenidos e incluir al menos 3 gráficas relevantes comparando las 3 propuestas.

**Rúbrica**: Se valorará la originalidad en la solución aportada, así como la capacidad de discutir los resultados de forma detallada. El requisito mínimo servirá para aprobar la actividad, bajo premisa de que la discusión del resultado sera apropiada.

IMPORTANTE:

* Si no se consigue una puntuación óptima, responder sobre la mejor puntuación obtenida.
* Para entrenamientos largos, recordad que podéis usar checkpoints de vuestros modelos para retomar los entrenamientos. En este caso, recordad cambiar los parámetros adecuadamente (sobre todo los relacionados con el proceso de exploración).
* Se deberá entregar unicamente el notebook y los pesos del mejor modelo en un fichero .zip, de forma organizada.
* Cada alumno deberá de subir la solución de forma individual.

## Imports, configuración y entorno

In [1]:
from __future__ import division

# Librerías estándar
# ===========================================================
import os
import sys
import types
import platform
import random
from pathlib import Path
import numpy as np

# Variables de entorno
# ===========================================================
from dotenv import load_dotenv

load_dotenv()

# keras-rl2 usa tensorflow.keras internamente: debe establecerse antes de importar TF
# ===========================================================
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Deep learning: TF
# ===========================================================
import tensorflow as tf

# Keras 2 (tf-keras): modelos, capas, optimizadores
# ===========================================================
import tf_keras
from tf_keras.models import Sequential
from tf_keras.layers import Dense, Activation, Flatten, Convolution2D, Permute
from tf_keras.optimizers.legacy import Adam
import tf_keras.backend as K

# Shim de compatibilidad: keras-rl2 1.0.5 usa tensorflow.python.keras (eliminado en TF 2.16+)
# ===========================================================
import tensorflow.keras as _tfkeras

if not hasattr(_tfkeras, "__version__"):
    _tfkeras.__version__ = tf_keras.__version__

_compat_generic = types.ModuleType("tensorflow.python.keras.utils.generic_utils")
_compat_generic.Progbar = tf_keras.utils.Progbar

_compat_utils = types.ModuleType("tensorflow.python.keras.utils")
_compat_utils.generic_utils = _compat_generic

_compat_keras = types.ModuleType("tensorflow.python.keras")
_compat_keras.callbacks = tf_keras.callbacks
_compat_keras.utils = _compat_utils

sys.modules.setdefault("tensorflow.python.keras", _compat_keras)
sys.modules.setdefault("tensorflow.python.keras.callbacks", tf_keras.callbacks)
sys.modules.setdefault("tensorflow.python.keras.utils", _compat_utils)
sys.modules.setdefault("tensorflow.python.keras.utils.generic_utils", _compat_generic)

# Entorno RL
# ===========================================================
import gym
from PIL import Image

# keras-rl2: agente DQN, políticas, memoria, callbacks
# ===========================================================
from rl.agents.dqn import DQNAgent
from rl.policy import LinearAnnealedPolicy, BoltzmannQPolicy, EpsGreedyQPolicy
from rl.memory import SequentialMemory
from rl.core import Processor
from rl.callbacks import Callback, FileLogger, ModelIntervalCheckpoint

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
IS_ARM_MAC = sys.platform == "darwin" and platform.machine() == "arm64"

print("TensorFlow version:", tf.__version__)
print("tf-keras version:  ", tf_keras.__version__)
print("ARM Mac:           ", IS_ARM_MAC)
print("GPUs disponibles:  ", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.18.0
tf-keras version:   2.18.0
ARM Mac:            True
GPUs disponibles:   [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Constantes

In [3]:
# Paths
# ===========================================================
BASE_FOLDER = Path(os.getenv("BASE_FOLDER"))
WEIGHTS_FOLDER = Path(os.getenv("WEIGHTS_FOLDER"))

# Reproducibilidad
# ===========================================================
SEED = int(os.getenv("SEED", 42))
np.random.seed(SEED)

# Entorno
# ===========================================================
ENV_NAME = "SpaceInvaders-v0"  # nombre del entorno: se usa en todo el código (pesos, logs, etc.)
SHAPE = (84, 84)
WINDOW_LENGTH = 4
INPUT_SHAPE = (WINDOW_LENGTH,) + SHAPE

# Red neuronal
# ===========================================================
DENSE_UNITS = 512
LEARNING_RATE = 0.00025

# Memoria replay
# ===========================================================
MEMORY_LIMIT = 1_000_000  # estándar DQN Atari

# Política eps-greedy con annealing
# ===========================================================
EPS_MAX = 1.0
EPS_MIN = 0.1
EPS_TEST = 0.05
EPS_ANNEALING_STEPS = 1_000_000  # eps decae de EPS_MAX a EPS_MIN durante estos pasos

# Agente DQN
# ===========================================================
NB_STEPS_WARMUP = 50_000  # pasos de exploración pura antes de empezar a aprender
GAMMA = 0.99  # factor de descuento para la recompensa futura
TARGET_MODEL_UPDATE = 10_000  # cada cuántos pasos se sincroniza la target network
TRAIN_INTERVAL = 4  # actualizar pesos cada N pasos de entorno
DELTA_CLIP = 1.0  # Huber loss clipping

# Entrenamiento
# ===========================================================
NB_STEPS = 2_000_000
CHECKPOINT_INTERVAL = 250_000
LOG_INTERVAL = 1_000
LOG_DISPLAY_INTERVAL = 10_000

# Test
# ===========================================================
NB_TEST_EPISODES = 10

# Inicializar entorno
# ===========================================================
env = gym.make(ENV_NAME)
env.seed(SEED)
nb_actions = env.action_space.n
print(f"Entorno: {ENV_NAME} | Acciones: {nb_actions}")


Entorno: SpaceInvaders-v0 | Acciones: 6


/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment SpaceInvaders-v0 is out of date. You should consider upgrading to version `v4`.
  logger.warn(
A.L.E: Arcade Learning Environment (version 0.7.5+db37282)
[Powered by Stella]
/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


## 1. Implementación red neuronal

In [4]:
class AtariProcessor(Processor):
    def process_observation(self, observation):
        assert observation.ndim == 3  # (height, width, channel)
        img = Image.fromarray(observation)
        img = img.resize(SHAPE).convert("L")  # SHAPE=(84,84), no INPUT_SHAPE=(4,84,84)
        processed_observation = np.array(img)
        assert processed_observation.shape == SHAPE
        return processed_observation.astype("uint8")

    def process_state_batch(self, batch):
        return batch.astype("float32") / 255.0

    def process_reward(self, reward):
        return np.clip(reward, -1.0, 1.0)

In [5]:
# Arquitectura DeepMind (Mnih et al. 2015) adaptada a SpaceInvaders
# INPUT_SHAPE = (4, 84, 84): keras-rl pasa el stack de frames en channels_first
# Permute reordena a channels_last (84, 84, 4) para que TF/cuDNN use sus kernels eficientemente
model = Sequential()
if K.image_data_format() == "channels_last":
    model.add(Permute((2, 3, 1), input_shape=INPUT_SHAPE))
elif K.image_data_format() == "channels_first":
    model.add(Permute((1, 2, 3), input_shape=INPUT_SHAPE))
else:
    raise RuntimeError("image_data_format desconocido.")

model.add(Convolution2D(32, (8, 8), strides=(4, 4), activation="relu"))
model.add(Convolution2D(64, (4, 4), strides=(2, 2), activation="relu"))
model.add(Convolution2D(64, (3, 3), strides=(1, 1), activation="relu"))
model.add(Flatten())
model.add(Dense(512, activation="relu"))
model.add(Dense(nb_actions, activation="linear"))

print(model.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 permute (Permute)           (None, 84, 84, 4)         0         
                                                                 
 conv2d (Conv2D)             (None, 20, 20, 32)        8224      
                                                                 
 conv2d_1 (Conv2D)           (None, 9, 9, 64)          32832     
                                                                 
 conv2d_2 (Conv2D)           (None, 7, 7, 64)          36928     
                                                                 
 flatten (Flatten)           (None, 3136)              0         
                                                                 
 dense (Dense)               (None, 512)               1606144   
                                                                 
 dense_1 (Dense)             (None, 6)                 3

## 2. Implementación del agente (DQN)

In [6]:
processor = AtariProcessor()

memory = SequentialMemory(limit=MEMORY_LIMIT, window_length=WINDOW_LENGTH)

policy = LinearAnnealedPolicy(
    EpsGreedyQPolicy(),
    attr="eps",
    value_max=EPS_MAX,
    value_min=EPS_MIN,
    value_test=EPS_TEST,
    nb_steps=EPS_ANNEALING_STEPS,
)

In [7]:
dqn = DQNAgent(
    model=model,
    nb_actions=nb_actions,
    policy=policy,
    memory=memory,
    processor=processor,
    nb_steps_warmup=NB_STEPS_WARMUP,
    gamma=GAMMA,
    target_model_update=TARGET_MODEL_UPDATE,
    train_interval=TRAIN_INTERVAL,
    delta_clip=DELTA_CLIP,
)
dqn.compile(Adam(learning_rate=LEARNING_RATE), metrics=["mae"])


2026-06-27 15:21:58.278712: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Pro
2026-06-27 15:21:58.278733: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 64.00 GB
2026-06-27 15:21:58.278740: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 25.92 GB
I0000 00:00:1782566518.278752 3085817 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1782566518.278770 3085817 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
I0000 00:00:1782566518.282211 3085817 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
2026-06-27 15:21:58.284367: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is

### 2.1 Entrenamiento del agente

In [8]:
class SaveBestWeights(Callback):
    """Guarda los pesos cuando se supera el mejor reward; incluye el step en el nombre."""

    def __init__(self, dirpath, env_name, verbose=1):
        super().__init__()
        self.dirpath = dirpath
        self.env_name = env_name
        self.verbose = verbose
        self.best_reward = -np.inf

    def on_episode_end(self, episode, logs):
        reward = logs.get("episode_reward", -np.inf)
        if reward > self.best_reward:
            prev = self.best_reward
            self.best_reward = reward
            step = self.model.step
            filepath = str(Path(self.dirpath) / f"dqn_{self.env_name}_best_step{step}.h5f")
            self.model.save_weights(filepath, overwrite=True)
            if self.verbose:
                print(f"\n[Best] Ep {episode} | step {step} | reward {reward:.1f} (prev {prev:.1f}) → {filepath}")


In [ ]:
# Crear estructura de carpetas
# ===========================================================
checkpoints_dir = WEIGHTS_FOLDER / "checkpoints"
best_dir = WEIGHTS_FOLDER / "best"
logs_dir = WEIGHTS_FOLDER / "logs"
final_dir = WEIGHTS_FOLDER / "final"
for d in [checkpoints_dir, best_dir, logs_dir, final_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Rutas de archivos
# ===========================================================
checkpoint_file = str(checkpoints_dir / f"dqn_{ENV_NAME}_step{{step}}.h5f")
log_file = str(logs_dir / f"dqn_{ENV_NAME}_log.json")

callbacks = [
    ModelIntervalCheckpoint(checkpoint_file, interval=CHECKPOINT_INTERVAL),
    FileLogger(log_file, interval=LOG_INTERVAL),
    SaveBestWeights(best_dir, ENV_NAME),
]

dqn.fit(env, nb_steps=NB_STEPS, callbacks=callbacks, log_interval=LOG_DISPLAY_INTERVAL, visualize=False)

final_file = str(final_dir / f"dqn_{ENV_NAME}_final_step{dqn.step}.h5f")
dqn.save_weights(final_file, overwrite=True)
print(f"Pesos finales: {final_file}")


Training for 2000000 steps ...
Interval 1 (0 steps performed)
   19/10000 [..............................] - ETA: 28s - reward: 0.0000e+00  

/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/tf_keras/src/engine/training_v1.py:2354: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
2026-06-27 15:21:58.482030: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_1/BiasAdd' id:125 op device:{requested: '', assigned: ''} def:{{{node dense_1/BiasAdd}} = BiasAdd[T=DT_FLOAT, _has_manual_control_dependencies=true, data_format="NHWC"](dense_1/MatMul, dense_1/BiasAdd/ReadVariableOp)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-06-27 15:21:58.489764: W tensorflow/c/c_api.cc:305] Operation '{name:'total_3/Assign' id:390 op device:{requested: '', assigned: ''} def:{{{node total_3/Assign}} = AssignVariableOp

  658/10000 [>.............................] - ETA: 20s - reward: 0.0122
[Best] Ep 0 | step 660 | reward 8.0 (prev -inf) → /Users/alejandro/projects/msc_ia/AR/trabajo/ar_proyecto_grupal_02/weights/best/dqn_SpaceInvaders-v0_best_step660.h5f
  742/10000 [=>............................] - ETA: 20s - reward: 0.0108

I0000 00:00:1782566520.025717 3085817 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1782566520.025730 3085817 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


 1305/10000 [==>...........................] - ETA: 18s - reward: 0.0146
[Best] Ep 1 | step 1306 | reward 11.0 (prev 8.0) → /Users/alejandro/projects/msc_ia/AR/trabajo/ar_proyecto_grupal_02/weights/best/dqn_SpaceInvaders-v0_best_step1306.h5f
 4432/10000 [============>.................] - ETA: 11s - reward: 0.0129
[Best] Ep 5 | step 4457 | reward 12.0 (prev 11.0) → /Users/alejandro/projects/msc_ia/AR/trabajo/ar_proyecto_grupal_02/weights/best/dqn_SpaceInvaders-v0_best_step4457.h5f
10000/10000 [==============================] - 20s 2ms/step - reward: 0.0117
14 episodes - episode_reward: 8.071 [5.000, 12.000] - lives: 2.089 - episode_frame_number: 1052.978 - frame_number: 14924.784

Interval 2 (10000 steps performed)
 1359/10000 [===>..........................] - ETA: 17s - reward: 0.0162
[Best] Ep 15 | step 11360 | reward 16.0 (prev 12.0) → /Users/alejandro/projects/msc_ia/AR/trabajo/ar_proyecto_grupal_02/weights/best/dqn_SpaceInvaders-v0_best_step11360.h5f
 5804/10000 [================>

2026-06-27 15:23:41.060388: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_1_1/BiasAdd' id:249 op device:{requested: '', assigned: ''} def:{{{node dense_1_1/BiasAdd}} = BiasAdd[T=DT_FLOAT, _has_manual_control_dependencies=true, data_format="NHWC"](dense_1_1/MatMul, dense_1_1/BiasAdd/ReadVariableOp)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-06-27 15:23:41.121374: W tensorflow/c/c_api.cc:305] Operation '{name:'loss_3/AddN' id:491 op device:{requested: '', assigned: ''} def:{{{node loss_3/AddN}} = AddN[N=2, T=DT_FLOAT, _has_manual_control_dependencies=true](loss_3/mul, loss_3/mul_1)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-06-27 15:23:4

  654/10000 [>.............................] - ETA: 2:01 - reward: 0.0122

### 2.2 Evaluación del modelo

In [ ]:
# Carga el best con mayor step disponible
best_files = sorted((WEIGHTS_FOLDER / "best").glob(f"dqn_{ENV_NAME}_best_step*.h5f"))
weights_file = str(best_files[-1])
print(f"Cargando: {weights_file}")

dqn.load_weights(weights_file)
dqn.test(env, nb_episodes=NB_TEST_EPISODES, visualize=False)


## 3. Conclusiones
Justificación de los parámetros seleccionados y de los resultados obtenidos

